# Multi-Output Structured Prediction

Predict four output fields from a short text input — a single signature with year, type, destination, and status outputs.

**Dataset:** `nasa_missions.json` — 100 NASA missions with name + description as inputs and four structured outputs: launch year, mission type, destination, status. (Sourced from public NASA mission descriptions, US-Government public domain.)

**What you'll learn:**
1. Define a multi-output signature (2 inputs → 4 outputs)
2. Write a composite metric that scores per-field accuracy
3. Validate signature + metric with `/validate-code` before submitting
4. Run inference remotely via the `/serve` endpoint

**Prerequisites:** Complete [01_quickstart.ipynb](01_quickstart.ipynb) first.

In [ ]:
import json
import os
from pathlib import Path

import dspy

from skynet_client import DSPyServiceClient, JobMonitor, serialize_source

In [ ]:
BASE_URL = os.getenv("DSPY_SERVICE_URL", "http://localhost:8000")
LM_BASE_URL = os.getenv("DSPY_LM_BASE_URL", "https://api.openai.com/v1")

# LM_API_KEY is the canonical name; OPENAI_API_KEY is accepted as a
# fallback for backwards-compat and the public OpenAI dev path.
LM_API_KEY = os.getenv("LM_API_KEY") or os.getenv("OPENAI_API_KEY")
if not LM_API_KEY:
    raise ValueError("Set LM_API_KEY (or OPENAI_API_KEY) — any non-empty token works for self-hosted gateways.")

MODEL_CONFIG = {
    "name": "openai/gpt-4o-mini",
    "base_url": LM_BASE_URL,
    "model_type": "responses",
    "temperature": 1.0,
    "max_tokens": 20000,
    "extra": {"api_key": LM_API_KEY},
}

dspy.configure(
    lm=dspy.LM(
        "openai/gpt-4o-mini",
        model_type="responses",
        temperature=1.0,
        max_tokens=20000,
        api_base=LM_BASE_URL,
        api_key=LM_API_KEY,
    )
)

client = DSPyServiceClient(BASE_URL)
client.health()

In [ ]:
# Browse available models in the catalog
models = client.models()
for m in models[:5]:
    print(f"  {m['name']} — {m.get('provider', '')}")

## 1. Load Dataset

Each row has 2 inputs (name, description) and 4 outputs (launch_year, mission_type, destination, status).

In [ ]:
with open(Path("../../data/nasa_missions.json")) as f:
    dataset = json.load(f)

print(f"{len(dataset)} missions")
print(f"Fields: {list(dataset[0].keys())}")
print()
sample = dataset[0]
for k, v in sample.items():
    print(f"  {k}: {v}")

## 2. Multi-Output Signature

Four output fields in a single signature — DSPy will fan them out into structured fields the LLM has to fill independently.

In [ ]:
class MissionClassifier(dspy.Signature):
    """Given a NASA mission name and one-sentence description, return launch year, type, destination, and status."""
    name: str = dspy.InputField(desc="Common mission name (e.g. \"Apollo 11\")")
    description: str = dspy.InputField(desc="One-sentence description of what the mission did")

    launch_year: int = dspy.OutputField(desc="Year of launch as an integer")
    mission_type: str = dspy.OutputField(desc="\"crewed\" or \"uncrewed\"")
    destination: str = dspy.OutputField(desc="Primary target body or orbit, lowercase (e.g. \"moon\", \"mars\", \"low_earth_orbit\")")
    status: str = dspy.OutputField(desc="One of: \"success\", \"partial_success\", \"failure\", \"active\"")


SIGNATURE_CODE = serialize_source(MissionClassifier)
print(SIGNATURE_CODE)

## 3. Composite Metric

For multi-output prediction, the metric needs to credit partial wins. Score each output field, average them, and report which fields agreed.

In [ ]:
# The metric is exec()'d server-side; helper imports stay inside the body.
METRIC_CODE = '''
def mission_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    fields = ["launch_year", "mission_type", "destination", "status"]
    hits = []
    misses = []
    for f in fields:
        g = getattr(gold, f, None)
        p = getattr(pred, f, None)
        if f == "launch_year":
            try:
                ok = int(p) == int(g)
            except (TypeError, ValueError):
                ok = False
        else:
            ok = str(g).strip().lower() == str(p).strip().lower()
        (hits if ok else misses).append(f)

    score = round(len(hits) / len(fields), 2)

    if not misses:
        return dspy.Prediction(score=1.0, feedback="All four fields correct.")

    detail = ", ".join(f"{f}: expected {getattr(gold, f)!r}, got {getattr(pred, f)!r}" for f in misses)
    feedback = f"Got {len(hits)}/{len(fields)}. Wrong: {detail}."
    return dspy.Prediction(score=score, feedback=feedback)
'''

## 4. Validate Before Submitting

The `/validate-code` endpoint catches missing fields and syntax errors before a job kicks off.

In [ ]:
validation = client.validate_code({
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "column_mapping": {
        "inputs": {"name": "name", "description": "description"},
        "outputs": {
            "launch_year": "launch_year",
            "mission_type": "mission_type",
            "destination": "destination",
            "status": "status",
        },
    },
})

print(f"Valid: {validation['valid']}")
if validation.get("errors"):
    print(f"Errors: {validation['errors']}")
if validation.get("warnings"):
    print(f"Warnings: {validation['warnings']}")
if validation.get("signature_fields"):
    print(f"Fields: {validation['signature_fields']}")

## 5. Submit Optimization

Note how each signature field maps to the corresponding JSON key in the dataset.

In [ ]:
payload = {
    "username": "notebook-user",
    "module_name": "dspy.ChainOfThought",
    "signature_code": SIGNATURE_CODE,
    "metric_code": METRIC_CODE,
    "optimizer_name": "dspy.GEPA",
    "optimizer_kwargs": {"auto": "light", "num_threads": 8},
    "compile_kwargs": {},
    "dataset": dataset,
    "column_mapping": {
        "inputs": {"name": "name", "description": "description"},
        "outputs": {
            "launch_year": "launch_year",
            "mission_type": "mission_type",
            "destination": "destination",
            "status": "status",
        },
    },
    "split_fractions": {"train": 0.5, "val": 0.3, "test": 0.2},
    "shuffle": True,
    "seed": 42,
    "model_config": MODEL_CONFIG,
    "reflection_model_config": MODEL_CONFIG,
}

optimization_id = client.submit(payload)
print(f"Submitted: {optimization_id}")

In [ ]:
monitor = JobMonitor(client, optimization_id)
result = monitor.poll(interval=5)

if result["status"] == "success":
    r = result["result"]
    print(f"\nBaseline score:  {r.get('baseline_test_metric', 'N/A')}")
    print(f"Optimized score: {r.get('optimized_test_metric', 'N/A')}")
    print(f"Runtime: {r.get('runtime_seconds', 0):.0f}s")
else:
    print(f"\nFailed: {result.get('message')}")

## 6. Serve the Optimized Program (Remote Inference)

Instead of downloading the artifact and running locally, you can call the `/serve` endpoint to run inference on the server. This is useful when you don't have `dspy` installed or want to share the endpoint.

In [ ]:
# Check serve availability
serve_info = client.serve_info(optimization_id)
serve_info

In [ ]:
test_missions = [
    {
        "name": "Mariner 9",
        "description": "First spacecraft to enter orbit around another planet, mapping Mars from above.",
    },
    {
        "name": "Apollo 11",
        "description": "First crewed mission to land humans on the Moon.",
    },
    {
        "name": "James Webb Space Telescope",
        "description": "Large infrared observatory operating at the Sun-Earth L2 Lagrange point.",
    },
]

for m in test_missions:
    response = client.serve(optimization_id, m)
    print(f"Mission: {m['name']}")
    print(f"  → {response}")
    print()

## 7. Local Inference (Alternative)

You can also download the program and run it locally with your own `dspy.LM`.

In [ ]:
program = client.load_program(optimization_id)

response = program(
    name="Voyager 1",
    description="Outer-planets probe that reached interstellar space and remains operational.",
)
print(f"  launch_year:  {response.launch_year}")
print(f"  mission_type: {response.mission_type}")
print(f"  destination:  {response.destination}")
print(f"  status:       {response.status}")

In [ ]:
dspy.inspect_history(n=1)

## What's Next

- **[04_advanced.ipynb](04_advanced.ipynb)** — Multi-input fusion + multi-label outputs (USPTO patents).
- **[05_gepa_showcase.ipynb](05_gepa_showcase.ipynb)** — GEPA on a hard reasoning task.
- **API Reference** — `http://localhost:8000/reference` for all endpoints including templates, analytics, and bulk operations.
- **Frontend Dashboard** — `http://localhost:3001` to view, compare, and manage optimizations visually.